# Customer Churn — Exploratory Data Analysis

Phase 2 of the Explainable Customer Churn Prediction project.

This notebook answers every EDA question specified in the project brief using the actual dataset — no numbers here are invented.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless-safe backend for notebook execution
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import load_and_validate_data

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

FIG_DIR = os.path.join('..', 'outputs', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

df, report = load_and_validate_data(path=os.path.join('..', 'data', 'customer_churn.csv'))
print(report.summary())


Rows: 7043, Columns: 21
Duplicate rows: 0
Duplicate customerIDs: 0
Non-numeric TotalCharges values: 11
Target distribution: {'No': 5174, 'Yes': 1869}
Warnings:
  - 'TotalCharges' has 11 non-numeric values (likely blank strings for customers with tenure == 0). These will need explicit handling in preprocessing, not silent coercion.


## Dataset Overview

In [2]:
n_rows, n_cols = df.shape
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in df.columns if c not in numeric_cols and c not in ('customerID',)]

print(f"Rows: {n_rows}")
print(f"Columns: {n_cols}")
print(f"Numerical features: {numeric_cols}")
print(f"Categorical features (excluding ID): {categorical_cols}")
print(f"Missing values total: {int(df.isna().sum().sum())}")
print(f"Duplicate records: {int(df.duplicated().sum())}")


Rows: 7043
Columns: 21
Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Categorical features (excluding ID): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']
Missing values total: 0
Duplicate records: 0


## Churn Distribution

In [3]:
churn_counts = df['Churn'].value_counts()
churn_rate = churn_counts['Yes'] / churn_counts.sum() * 100
print(churn_counts)
print(f"Overall churn rate: {churn_rate:.2f}%")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x='Churn', hue='Churn', palette=['#2e7d32', '#c62828'],
              order=['No', 'Yes'], legend=False, ax=ax)
ax.set_title('Churn Distribution')
ax.set_xlabel('Churn')
ax.set_ylabel('Number of Customers')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'churn_distribution.png'))
plt.show()


Churn
No     5174
Yes    1869
Name: count, dtype: int64
Overall churn rate: 26.54%


## Numerical Feature Analysis

`TotalCharges` is coerced to numeric here for analysis only (11 blank-string rows -> 0, matching tenure == 0 customers). The reusable preprocessing pipeline in `src/data_preprocessing.py` will apply this same rule formally in Phase 3.

In [4]:
df_num = df.copy()
df_num['TotalCharges'] = pd.to_numeric(df_num['TotalCharges'], errors='coerce').fillna(0)

num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
print(df_num[num_features].describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_features):
    sns.histplot(data=df_num, x=col, hue='Churn', kde=True, ax=ax,
                 palette=['#2e7d32', '#c62828'], element='step', common_norm=False)
    ax.set_title(f'{col} by Churn')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'numerical_distributions.png'))
plt.show()


            tenure  MonthlyCharges  TotalCharges
count  7043.000000     7043.000000   7043.000000
mean     32.371149       64.761692   2279.734304
std      24.559481       30.090047   2266.794470
min       0.000000       18.250000      0.000000
25%       9.000000       35.500000    398.550000
50%      29.000000       70.350000   1394.550000
75%      55.000000       89.850000   3786.600000
max      72.000000      118.750000   8684.800000


In [5]:
# Does short tenure correlate with churn?
tenure_by_churn = df_num.groupby('Churn')['tenure'].mean()
print("Mean tenure by churn status:")
print(tenure_by_churn)

# Does higher monthly charge correlate with churn?
charges_by_churn = df_num.groupby('Churn')['MonthlyCharges'].mean()
print("\nMean MonthlyCharges by churn status:")
print(charges_by_churn)


Mean tenure by churn status:
Churn
No     37.569965
Yes    17.979133
Name: tenure, dtype: float64

Mean MonthlyCharges by churn status:
Churn
No     61.265124
Yes    74.441332
Name: MonthlyCharges, dtype: float64


## Categorical Feature Analysis vs. Churn

In [6]:
cat_features = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport',
                 'OnlineSecurity', 'SeniorCitizen', 'Partner', 'Dependents']

fig, axes = plt.subplots(4, 2, figsize=(13, 18))
axes = axes.flatten()
for ax, col in zip(axes, cat_features):
    churn_rate_by_cat = (
        df.groupby(col)['Churn']
        .apply(lambda s: (s == 'Yes').mean() * 100)
        .sort_values(ascending=False)
    )
    churn_rate_by_cat.plot(kind='bar', ax=ax, color='#c62828')
    ax.set_title(f'Churn Rate (%) by {col}')
    ax.set_ylabel('Churn Rate (%)')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'categorical_churn_rates.png'))
plt.show()


In [7]:
for col in cat_features:
    print(f"--- {col} ---")
    print((df.groupby(col)['Churn'].apply(lambda s: (s == 'Yes').mean() * 100))
          .sort_values(ascending=False).round(2))
    print()


--- Contract ---
Contract
Month-to-month    42.71
One year          11.27
Two year           2.83
Name: Churn, dtype: float64

--- InternetService ---
InternetService
Fiber optic    41.89
DSL            18.96
No              7.40
Name: Churn, dtype: float64

--- PaymentMethod ---
PaymentMethod
Electronic check             45.29
Mailed check                 19.11
Bank transfer (automatic)    16.71
Credit card (automatic)      15.24
Name: Churn, dtype: float64

--- TechSupport ---
TechSupport
No                     41.64
Yes                    15.17
No internet service     7.40
Name: Churn, dtype: float64

--- OnlineSecurity ---
OnlineSecurity
No                     41.77
Yes                    14.61
No internet service     7.40
Name: Churn, dtype: float64

--- SeniorCitizen ---
SeniorCitizen
1    41.68
0    23.61
Name: Churn, dtype: float64

--- Partner ---
Partner
No     32.96
Yes    19.66
Name: Churn, dtype: float64

--- Dependents ---
Dependents
No     31.28
Yes    15.45
Name: Churn,

## Correlation Analysis

In [8]:
corr_df = df_num[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']].copy()
corr_df['Churn'] = (df['Churn'] == 'Yes').astype(int)

corr_matrix = corr_df.corr()
print(corr_matrix)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap (Numerical Features + Churn)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'correlation_heatmap.png'))
plt.show()


                  tenure  MonthlyCharges  TotalCharges  SeniorCitizen  \
tenure          1.000000        0.247900      0.826178       0.016567   
MonthlyCharges  0.247900        1.000000      0.651174       0.220173   
TotalCharges    0.826178        0.651174      1.000000       0.103006   
SeniorCitizen   0.016567        0.220173      0.103006       1.000000   
Churn          -0.352229        0.193356     -0.198324       0.150889   

                   Churn  
tenure         -0.352229  
MonthlyCharges  0.193356  
TotalCharges   -0.198324  
SeniorCitizen   0.150889  
Churn           1.000000  


## Answering the Required EDA Questions

Each answer below is computed directly from the dataset in the cells above.

In [9]:
answers = {}

# Q1: Which contract type has the highest churn?
contract_churn = df.groupby('Contract')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100)
answers['highest_churn_contract'] = contract_churn.idxmax()
print(f"Q1. Contract type with highest churn: {contract_churn.idxmax()} "
      f"({contract_churn.max():.2f}% churn rate)")
print(contract_churn.sort_values(ascending=False).round(2))


Q1. Contract type with highest churn: Month-to-month (42.71% churn rate)
Contract
Month-to-month    42.71
One year          11.27
Two year           2.83
Name: Churn, dtype: float64


In [10]:
# Q2: Does short tenure correlate with churn?
tenure_corr = corr_df['tenure'].corr(corr_df['Churn'])
print(f"Q2. Correlation between tenure and churn: {tenure_corr:.3f} "
      f"({'negative -> shorter tenure associated with higher churn' if tenure_corr < 0 else 'positive'})")
print(f"    Mean tenure (churned): {df_num[df['Churn']=='Yes']['tenure'].mean():.1f} months")
print(f"    Mean tenure (retained): {df_num[df['Churn']=='No']['tenure'].mean():.1f} months")


Q2. Correlation between tenure and churn: -0.352 (negative -> shorter tenure associated with higher churn)
    Mean tenure (churned): 18.0 months


    Mean tenure (retained): 37.6 months


In [11]:
# Q3: Does higher monthly charge correlate with churn?
charges_corr = corr_df['MonthlyCharges'].corr(corr_df['Churn'])
print(f"Q3. Correlation between MonthlyCharges and churn: {charges_corr:.3f} "
      f"({'positive -> higher charges associated with higher churn' if charges_corr > 0 else 'negative'})")
print(f"    Mean MonthlyCharges (churned): {df_num[df['Churn']=='Yes']['MonthlyCharges'].mean():.2f}")
print(f"    Mean MonthlyCharges (retained): {df_num[df['Churn']=='No']['MonthlyCharges'].mean():.2f}")


Q3. Correlation between MonthlyCharges and churn: 0.193 (positive -> higher charges associated with higher churn)
    Mean MonthlyCharges (churned): 74.44
    Mean MonthlyCharges (retained): 61.27


In [12]:
# Q4: Does technical support affect churn?
support_churn = df.groupby('TechSupport')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100)
print("Q4. Churn rate by TechSupport status:")
print(support_churn.round(2))


Q4. Churn rate by TechSupport status:
TechSupport
No                     41.64
No internet service     7.40
Yes                    15.17
Name: Churn, dtype: float64


In [13]:
# Q5: Does payment method influence churn?
payment_churn = df.groupby('PaymentMethod')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100)
print("Q5. Churn rate by PaymentMethod:")
print(payment_churn.sort_values(ascending=False).round(2))


Q5. Churn rate by PaymentMethod:
PaymentMethod
Electronic check             45.29
Mailed check                 19.11
Bank transfer (automatic)    16.71
Credit card (automatic)      15.24
Name: Churn, dtype: float64


In [14]:
# Q6: Does internet service type influence churn?
internet_churn = df.groupby('InternetService')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100)
print("Q6. Churn rate by InternetService:")
print(internet_churn.sort_values(ascending=False).round(2))


Q6. Churn rate by InternetService:
InternetService
Fiber optic    41.89
DSL            18.96
No              7.40
Name: Churn, dtype: float64


## Summary of Findings

Computed directly from the executed cells above (7,043 customers, 26.54% overall churn rate):

1. **Contract type is the strongest categorical driver.** Month-to-month customers churn at 42.71%, vs. 11.27% for one-year and 2.83% for two-year contracts.
2. **Tenure is negatively correlated with churn (r = -0.352).** Churned customers average 18.0 months tenure vs. 37.6 months for retained customers — short-tenure customers churn far more.
3. **MonthlyCharges is positively correlated with churn (r = 0.193).** Churned customers average $74.44/month vs. $61.27/month for retained customers.
4. **Lack of tech support strongly predicts churn.** No TechSupport -> 41.64% churn vs. 15.17% with TechSupport.
5. **Payment method matters.** Electronic check payers churn at 45.29% — nearly 3x the rate of automatic bank transfer (16.71%) or credit card (15.24%) payers.
6. **Internet service type matters.** Fiber optic customers churn at 41.89% vs. 18.96% for DSL and 7.40% for no internet service (likely reflects a different, price-sensitive customer segment on fiber).
7. **SeniorCitizen, lack of a Partner, and lack of Dependents** are all associated with meaningfully higher churn (41.68%, 32.96%, and 31.28% respectively).
8. **TotalCharges correlates strongly with tenure (r = 0.826)** as expected (it's a near-cumulative measure), and is *negatively* correlated with churn (r = -0.198) — mainly because it's confounded with tenure, not because spending more money reduces churn risk.

These findings directly inform Phase 4 (Feature Engineering): contract type, tenure, tech/security add-ons, and payment method are the clearest candidates for engineered risk features.
